<div style="padding:20px; border-radius:6px; background-color:#ffe6e6; color:#b30000; border-left:5px solid #b30000;">
<b>To be able to run this notebook, choose environment <code>mandala_env</code></b>
</div>

In [ ]:
import os
import yaml
import pandas as pd
import datetime
import mandala
import fsspec
import xarray as xr
import zcollection as zc
import zcollection.fs_utils as zc_fs
from typing import Literal

from training_ofs import load_s3_options_from_vault

## Introduction

Marine Data Lake (Mandala) stores MMS1 observational and numerical model datasets in object storage. These datasets can be accessed through the S3 protocol.

### s3 configurations

In [ ]:
# Paste your Vault token here for local runs.
VAULT_TOKEN = ""

s3_options = load_s3_options_from_vault(token=VAULT_TOKEN)

fs = fsspec.filesystem(
    's3',
    key=s3_options['key'],
    secret=s3_options['secret'],
    client_kwargs=s3_options['client_kwargs'],
)
fs


### Listing available files/directories

In [ ]:
files = fs.ls('mandala-prod1/datalake/model')
files

In [ ]:
# model
fs.ls('mandala-prod1/datalake/model/collections')

In [ ]:
# obs
fs.ls('mandala-prod1/datalake/obs/collections')

### Using `mandala`

<div style="padding:2px; border-radius:2px; background-color:#ffe6e6; color:#b30000; border-left:5px solid #b30000;">> There is a bug, need to be updated. Use <a href="#Using-zcollection">zcollection</a> directly in the meantime</div>

#### Get catalog

In [ ]:
def get_catalog(database: Literal['obs','model'], s3_options: dict):
    catalog_path = os.path.join('mandala-prod1/datalake/',database,'catalog.json')
    return mandala.MandalaCatalog(path = catalog_path,
                          protocol="s3",
                          options = s3_options)

In [ ]:
def get_catalog(database: Literal['obs','model'], s3_options: dict):
    catalog_path = os.path.join('mandala-prod1/datalake/',database,'catalog_hindcast.json')
    return mandala.MandalaCatalog(path = catalog_path,
                          protocol="s3",
                          options = s3_options)

In [ ]:
# observation
catalogobs = get_catalog('obs', s3_options)
catalogobs

In [ ]:
# model
catalogmodel = get_catalog('model', s3_options)
catalogmodel

#### Processing datasets

All datasets are stores in `zcollection` format, which is built on top of the `zarr` format. Therefore, they can be efficiently processed using `xarray`.

Here is an example processing dataset `RADAR_TOTAL_BADA`. Other datasets can be accessed and processed in the same way by changing the dataset path.

In [ ]:
bada = catalogobs.open_collection(path='mandala-prod1/datalake/model/collections/inacawo_hindcast_ocean_2d').load()
bada

In [ ]:
# load to xarray.Dataset
bada = bada.to_xarray()
bada

In [ ]:
# quick plot
bada.u.isel(time=10).plot()

### Using `zcollection`

In [ ]:
fs.invalidate_cache() 
cawoflow = "mandala-prod1/datalake/model/collections/inacawo_wrf_2d"
z_cwflow = zc_fs.get_fs(filesystem=fs)
zcwflow_dset = zc.open_collection(
    path=cawoflow, mode="r", filesystem=z_cwflow
)
zcwflow_dset
date_filter = (
    "(year == 2026 and month == 9 and day >= 12) or "
    "(year == 2026 and month == 9 and day <= 13)"
)

dset = zcwflow_dset.load(delayed=True, filters=date_filter).to_xarray()
dset

In [ ]:
fs.invalidate_cache() 
cawoflow = "mandala-prod1/datalake/model/collections/inacawo_wrf_3d"
z_cwflow = zc_fs.get_fs(filesystem=fs)
zcwflow_dset = zc.open_collection(
    path=cawoflow, mode="r", filesystem=z_cwflow
)
zcwflow_dset
date_filter = (
    "(year == 2026 and month == 9 and day >= 12) or "
    "(year == 2026 and month == 9 and day <= 13)"
)

dset = zcwflow_dset.load(delayed=True, filters=date_filter).to_xarray()
dset

In [ ]:
# quick plot 

dset.isel(TSTEP=10).sig_wave_height.plot()